# Lesson 2 — Prompt templates and chains (the `|` pipe)

**Goal:** stop hard-coding prompts. Build a reusable template with blanks,
then connect `template -> model -> text` into one callable *chain*.

## Two concepts

**1. PromptTemplate** — a prompt with named blanks `{like_this}` you fill at
call time. The clean version of f-string prompt-building.

**2. The pipe `|` (LCEL)** — connect pieces left-to-right like a Unix pipe:
`prompt | model | parser`. The whole thing becomes one *chain*. Each piece is
a *Runnable* (anything with `.invoke()`), which is why they snap together.

### The proxy parallel — `api/proxy/database.py:259-283`
Your `_generate_summary` is a **second, hand-rolled model call**: it f-strings
a prompt, wraps it in a payload, POSTs, then digs out
`['choices'][0]['message']['content']`. Three separate steps. The chain below
is those same three steps joined into one.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(
    base_url='http://127.0.0.1:8080/v1',
    api_key='not-needed',
    model='qwen3-14b-q4_k_m',
    temperature=0.3,
)

### Piece 1 — the template
`{language}` and `{concept}` are blanks. The template is reusable: same
structure, different fill-ins each call.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a {language} tutor. Explain clearly for a beginner.'),
    ('human', 'Explain this concept with a tiny code example: {concept}'),
])

### Pieces 2 & 3 — model + parser, joined with `|`
`StrOutputParser` pulls the `.content` string out of the model's reply, so the
chain's final output is plain text.

In [ ]:
chain = prompt | model | StrOutputParser()
chain

### Call the chain
We supply only the blanks. The chain formats the prompt, calls the model, and
parses the reply.

In [ ]:
result = chain.invoke({'language': 'Python', 'concept': 'a list comprehension'})
print(result)

---
**Try it:** run the chain again with different blanks — same chain, new
question:
```python
chain.invoke({'language': 'JavaScript', 'concept': 'async/await'})
```

In [ ]:
chain.invoke({'language': 'JavaScript', 'concept': 'async/await'})